In [0]:
%python
'''
In this notebook, I have created two tables   IN   delta_catalog.delta_schema :
 1) customer_delta_bronze1 and 
 2) customer_delta_bronze_2

IN bronze1-> i created normal table adn inserted records from file using insert from file
IN bronze2-> i used CTAS command to create table 

I 1st bronze table, tried to use COPY INTO COMMAND,  it will load all the files including the data we inserted before. Bcz before we had inserted recods from normal insert.
In 2nd bronze table, tried to use COPY INTO COMMAND for first time for 1 file, it worked, next time while running the COPY INTO COMMAND, it skipped that file because COPY OPTIONS ('force' = 'false') was set. And third time I added new file with new column(schema change)  and set --> FORMAT OPTIONS ('mergeSchema' = 'true') and  --> COPY OPTIONS (mergeSchema=True) was set. Then it automatically added that schema to target table with everything null.

1. FORMAT_OPTIONS('mergeSchema'='true')
This tells Databricks how to interpret the source files.
If multiple CSV files have different schemas, Databricks merges the source schemas it discovers.

2. COPY_OPTIONS('mergeSchema'='true')
This tells Databricks to evolve the target Delta table schema.

'''



In [0]:
USE CATALOG delta_catalog;

In [0]:
 CREATE TABLE demo_schema.customer_delta_bronze1 (
     Customerid INT,
     Surname STRING,
     Creditscore INT,
     Geography STRING,
     Gender STRING,
     Age INT,
     Tenure INT,
     Balance DOUBLE,
     Estimatedsalary DOUBLE
 )
 USING DELTA
 LOCATION 'abfss://testdatabricksdemo@datbricksstrgaccextdl.dfs.core.windows.net/DemoCopyinto/schema/customer_delta_bronze1';

In [0]:
%sql
INSERT INTO demo_schema.customer_delta_bronze1
SELECT 
  CAST(Customerid AS INT) AS Customerid,
  Surname,
  CAST(Creditscore AS INT) AS Creditscore,
  Geography,
  Gender,
  CAST(Age AS INT) AS Age,
  CAST(Tenure AS INT) AS Tenure,
  CAST(Balance AS DOUBLE) AS Balance,
  CAST(Estimatedsalary AS DOUBLE) AS Estimatedsalary
FROM read_files(
  'abfss://testdatabricksdemo@datbricksstrgaccextdl.dfs.core.windows.net/bankchurn/bankchurn_1_50.csv',
  format => 'csv',
  header => 'true'
);

In [0]:
select * from demo_schema.customer_delta_bronze1;

In [0]:
CREATE TABLE demo_schema.customer_delta_bronze_2
USING DELTA
LOCATION "abfss://testdatabricksdemo@datbricksstrgaccextdl.dfs.core.windows.net/DemoCopyinto/schema/customer_delta_bronze_2"
AS
SELECT
    CAST(Customerid AS INT) AS Customerid,
    Surname,
    CAST(Creditscore AS INT) AS Creditscore,
    Geography,
    Gender,
    CAST(Age AS INT) AS Age,
    CAST(Tenure AS INT) AS Tenure,
    CAST(Balance AS DOUBLE) AS Balance,
    CAST(Estimatedsalary AS DOUBLE) AS Estimatedsalary
FROM read_files(
  'abfss://testdatabricksdemo@datbricksstrgaccextdl.dfs.core.windows.net/bankchurn/bankchurn_1_50.csv',
  format => 'csv',
  header => 'true'
);

In [0]:
SELECT * FROM demo_schema.customer_delta_bronze_2;

In [0]:
COPY INTO demo_schema.customer_delta_bronze1
FROM 'abfss://testdatabricksdemo@datbricksstrgaccextdl.dfs.core.windows.net/bankchurn'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' =  'true', 'inferSchema' = 'true')
COPY_OPTIONS ('force' = 'false')

In [0]:
select * from demo_schema.customer_delta_bronze1;

In [0]:
COPY INTO delta_catalog.demo_schema.customer_delta_bronze_2
FROM 'abfss://testdatabricksdemo@datbricksstrgaccextdl.dfs.core.windows.net/bankchurn'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true',
                'inferSchema' = 'true',
                'mergeSchema' = 'true')
COPY_OPTIONS('force'= 'false', 'mergeSchema' = 'true');

In [0]:
COPY INTO delta_catalog.demo_schema.customer_delta_bronze_2
FROM 'abfss://testdatabricksdemo@datbricksstrgaccextdl.dfs.core.windows.net/bankchurn'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true',
                'inferSchema' = 'true')
COPY_OPTIONS('force'= 'false');

In [0]:
COPY INTO delta_catalog.demo_schema.customer_delta_bronze_2
FROM 'abfss://testdatabricksdemo@datbricksstrgaccextdl.dfs.core.windows.net/bankchurn'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true',
                'inferSchema' = 'true')
COPY_OPTIONS('force'= 'true');

In [0]:
select count(*) from delta_catalog.demo_schema.customer_delta_bronze_2

In [0]:
COPY INTO delta_catalog.demo_schema.customer_delta_bronze_2
FROM (
    SELECT CAST(_c0 AS INT) AS Customerid,
           _c1 AS Surname,
           CAST(_c2 AS INT) AS Creditscore,
           _c3 AS Geography,
           _c4 AS Gender,
           CAST(_c5 AS INT) AS Age,
           CAST(_c6 AS INT) AS Tenure,
           CAST(_c7 AS DOUBLE) AS Balance,
           CAST(_c8 AS DOUBLE) AS Estimatedsalary,
           CAST(_c9 AS INT) AS NumOfProducts
FROM 'abfss://testdatabricksdemo@datbricksstrgaccextdl.dfs.core.windows.net/bankchurn')
FILEFORMAT = CSV
FORMAT_OPTIONS (
    'header' = 'false',
                'inferSchema' = 'true',
                );

In [0]:
DESCRIBE HISTORY delta_catalog.demo_schema.customer_delta_bronze_2

In [0]:
RESTORE TABLE delta_catalog.demo_schema.customer_delta_bronze_2 TO VERSION AS OF 3;

In [0]:
SELECT * FROM delta_catalog.demo_schema.customer_delta_bronze_2

In [0]:
SELECT * FROM delta_catalog.demo_schema.customer_delta_bronze_2 WHERE NumOfProducts is not NULL

In [0]:
DESCRIBE EXTENDED delta_catalog.demo_schema.customer_delta_bronze_2

In [0]:
DESCRIBE DETAIL delta_catalog.demo_schema.customer_delta_bronze_2;